## Data Base Testing ground

This notebook is intended for testing purposes, primarily to explore relationships and potential database design approaches. 

It serves as a template; the actual database and its data will be implemented in PostgreSQL.

In [1]:
%load_ext sql
%sql sqlite:////dsa/groups/casestudycf25/team02/casestudycf25t02.sqlite.db

'Connected: @/dsa/groups/casestudycf25/team02/casestudycf25t02.sqlite.db'

In [2]:
%%sql 
SELECT name 
FROM sqlite_master 
WHERE type='table';

 * sqlite:////dsa/groups/casestudycf25/team02/casestudycf25t02.sqlite.db
Done.


name


In [42]:
%%sql
PRAGMA table_info(Dim_NPI);

 * sqlite:////dsa/groups/casestudycf25/team02/casestudycf25t02.sqlite.db
Done.


cid,name,type,notnull,dflt_value,pk
0,NPI,BIGINT,0,None,1
1,Specialty,TEXT,0,None,0
2,Name,DATE,0,None,0
3,Primary_Address,TEXT,0,None,0
4,Zipcode,BIGINT,0,None,0


In [43]:
%%sql
PRAGMA foreign_key_list(Dim_NPI);

 * sqlite:////dsa/groups/casestudycf25/team02/casestudycf25t02.sqlite.db
Done.


id,seq,table,from,to,on_update,on_delete,match
0,0,Dim_Census_per_County_Tract,Zipcode,ZIP_Code,NO ACTION,NO ACTION,NONE


-------------------------------------------------------------------------------------------------------------------------------

## Delete tables

In [44]:
%%sql
DROP TABLE IF EXISTS Fact_DMEPOS_Ref_Provider_and_Service;
DROP TABLE IF EXISTS Fact_DMEPOS_Supplier_and_Service;

DROP TABLE IF EXISTS Dim_NPI_Year;
DROP TABLE IF EXISTS Dim_OIG_Exclusion_List;
DROP TABLE IF EXISTS Dim_NPI;
DROP TABLE IF EXISTS Dim_Medicare_Enrollment;
DROP TABLE IF EXISTS Dim_Census_per_County_Tract;
DROP TABLE IF EXISTS Dim_Location_Details;
DROP TABLE IF EXISTS Dim_Dates;
DROP TABLE IF EXISTS Dim_Service_Grouping;
DROP TABLE IF EXISTS Dim_Service;

 * sqlite:////dsa/groups/casestudycf25/team02/casestudycf25t02.sqlite.db
Done.
Done.
Done.
Done.
Done.
Done.
Done.
Done.
Done.
Done.
Done.


[]

In [45]:
%%sql
DROP TABLE IF EXISTS Dim_CMS_Open_Payments_Manufacturer;
DROP TABLE IF EXISTS Fact_CMS_Open_Payment_General;
DROP TABLE IF EXISTS Fact_CMS_Open_Payment_Ownership;

 * sqlite:////dsa/groups/casestudycf25/team02/casestudycf25t02.sqlite.db
Done.
Done.
Done.


[]

## Dim Service

In [13]:
%%sql
CREATE TABLE Dim_Service (
    HCPCS_cd VARCHAR(50) PRIMARY KEY,
    HCPCS_desc TEXT
);


 * sqlite:////dsa/groups/casestudycf25/team02/casestudycf25t02.sqlite.db
Done.


[]

## Dim Service Grouping

In [14]:
%%sql
CREATE TABLE Dim_Service_Grouping (
    HCPCS_cd VARCHAR(50),
    RBCS_id VARCHAR(50),
    RBCS_desc TEXT,
    RBCS_Lvl TEXT,
    FOREIGN KEY (HCPCS_cd) REFERENCES Dim_Service(HCPCS_cd)
);

 * sqlite:////dsa/groups/casestudycf25/team02/casestudycf25t02.sqlite.db
Done.


[]

## Dim Dates

In [16]:
%%sql
CREATE TABLE Dim_Dates (
    Date_Key BIGINT PRIMARY KEY,
    Year INT
);

 * sqlite:////dsa/groups/casestudycf25/team02/casestudycf25t02.sqlite.db
Done.


[]

## Dim Location Details

In [20]:
%%sql
CREATE TABLE Dim_Location_Details (
    ZIP_Code BIGINT,
    County_Tract VARCHAR(50),
    County VARCHAR(100),
    City VARCHAR(100),
    State VARCHAR(50),
    PRIMARY KEY (ZIP_Code,County_Tract,County )
);

 * sqlite:////dsa/groups/casestudycf25/team02/casestudycf25t02.sqlite.db
Done.


[]

## Dim Census Per County Tract

In [21]:
%%sql
CREATE TABLE Dim_Census_per_County_Tract (
    ZIP_Code BIGINT,
    County_Tract VARCHAR(50),
    Avg_Household_Income FLOAT,
    Num_Bachelor_Degree_H INT,
    PovertyRate FLOAT,
    Num_of_Pop INT,
    Average_Age FLOAT,
    Num_of_Married INT,
    Num_Medicaid_Holder INT,
    Num_of_Insured INT,
    Average_Family_Size FLOAT,
    PRIMARY KEY (ZIP_Code, County_Tract),
    FOREIGN KEY (ZIP_Code) REFERENCES Dim_Location_Details(ZIP_Code),
    FOREIGN KEY (County_Tract) REFERENCES Dim_Location_Details(County_Tract)
);

 * sqlite:////dsa/groups/casestudycf25/team02/casestudycf25t02.sqlite.db
Done.


[]

## DIM Medicare Enrollment

In [22]:
%%sql
CREATE TABLE Dim_Medicare_Enrollment (
    County VARCHAR(100),
    State VARCHAR(50),
    Year INT,
    Part_A_Enrollment INT,
    Part_B_Enrollment INT,
    Total_Enrollment INT,
    PRIMARY KEY (County, State, Year),
    FOREIGN KEY (County) REFERENCES Dim_Location_Details(County)
);

 * sqlite:////dsa/groups/casestudycf25/team02/casestudycf25t02.sqlite.db
Done.


[]

## DIM NPI

In [23]:
%%sql
CREATE TABLE Dim_NPI (
    NPI BIGINT PRIMARY KEY,
    Specialty TEXT,
    Name DATE,
    Primary_Address TEXT,
    Zipcode BIGINT,
    FOREIGN KEY (Zipcode) REFERENCES Dim_Census_per_County_Tract(ZIP_Code)
);


 * sqlite:////dsa/groups/casestudycf25/team02/casestudycf25t02.sqlite.db
Done.


[]

## DIM OIG Exclusion List

In [30]:
%%sql
CREATE TABLE Dim_OIG_Exclusion_List (
    NPI BIGINT PRIMARY KEY,
    Type_of_Exclusion VARCHAR(255),
    Max_Exclusion_Date DATE,
    FOREIGN KEY (NPI) REFERENCES Dim_NPI(NPI)
);

 * sqlite:////dsa/groups/casestudycf25/team02/casestudycf25t02.sqlite.db
Done.


[]

## DIM NPI And Year

In [31]:
%%sql
CREATE TABLE Dim_NPI_Year (
    Year INT,
    NPI INT,
    speciality_desc TEXT,
    Bene_Avg_Age FLOAT,
    Bene_Feml_Cnt FLOAT,
    Bene_Male_Cnt FLOAT,
    Bene_Race_Wht_Cnt FLOAT,
    Bene_Race_Black_Cnt FLOAT,
    Bene_Race_Api_Cnt FLOAT,
    Bene_Race_Hspnc_Cnt FLOAT,
    Bene_Race_Natind_Cnt FLOAT,
    Bene_Race_Othr_Cnt FLOAT,
    Bene_Ndual_Cnt FLOAT,
    Bene_Dual_Cnt FLOAT,
    Bene_CC_BH_ADHD_OthCD_V1_Pct FLOAT,
    Bene_CC_BH_Alcohol_Drug_V1_Pct FLOAT,
    Bene_CC_BH_Tobacco_V1_Pct FLOAT,
    Bene_CC_BH_Alz_NonAlzdem_V2_Pct FLOAT,
    Bene_CC_BH_Anxiety_V1_Pct FLOAT,
    Bene_CC_BH_Bipolar_V1_Pct FLOAT,
    Bene_CC_BH_Mood_V2_Pct FLOAT,
    Bene_CC_BH_Depress_V1_Pct FLOAT,
    Bene_CC_BH_PD_V1_Pct FLOAT,
    Bene_CC_BH_PTSD_V1_Pct FLOAT,
    Bene_CC_BH_Schizo_OthPsy_V1_Pct FLOAT,
    Bene_CC_PH_Asthma_V2_Pct FLOAT,
    Bene_CC_PH_Afib_V2_Pct FLOAT,
    Bene_CC_PH_Cancer6_V2_Pct FLOAT,
    Bene_CC_PH_CKD_V2_Pct FLOAT,
    Bene_CC_PH_COPD_V2_Pct FLOAT,
    Bene_CC_PH_Diabetes_V2_Pct FLOAT,
    Bene_CC_PH_HF_NonIHD_V2_Pct FLOAT,
    Bene_CC_PH_Hyperlipidemia_V2_Pct FLOAT,
    Bene_CC_PH_Hypertension_V2_Pct FLOAT,
    Bene_CC_PH_IschemicHeart_V2_Pct FLOAT,
    Bene_CC_PH_Parkinson_V2_Pct FLOAT,
    Bene_CC_PH_Arthritis_V2_Pct FLOAT,
    Bene_CC_PH_Stroke_TIA_V2_Pct FLOAT,
    Bene_Avg_Risk_Scre FLOAT,
    PRIMARY KEY (Year, NPI),
    FOREIGN KEY (NPI) REFERENCES Dim_NPI(NPI)
);

 * sqlite:////dsa/groups/casestudycf25/team02/casestudycf25t02.sqlite.db
Done.


[]

## DIM DMEPOS Supplier and Service

In [33]:
%%sql
CREATE TABLE Fact_DMEPOS_Supplier_and_Service (
    NPI BIGINT,
    Year INT,
    Zip_Code BIGINT,
    HCPCS_Code VARCHAR(50),
    Rental_Ind BOOLEAN,
    Suppressed_Ind BOOLEAN,
    Tot_Ben FLOAT,
    Tot_Claims FLOAT,
    Total_Serv FLOAT,
    Avg_Sbmt_Chrg FLOAT,
    Avg_Mc_Allwd_Amt FLOAT,
    Avg_Mc_Pymt_Amt FLOAT,
    Std_Mc_Pymt_Amt FLOAT,
    PRIMARY KEY (NPI, Year),
    FOREIGN KEY (HCPCS_Code) REFERENCES Dim_Service(HCPCS_cd),
    FOREIGN KEY (NPI) REFERENCES Dim_NPI_Year(NPI)
);

 * sqlite:////dsa/groups/casestudycf25/team02/casestudycf25t02.sqlite.db
Done.


[]

## FACT DMEPOS Refferal Provider and Service

In [35]:
%%sql
CREATE TABLE Fact_DMEPOS_Ref_Provider_and_Service (
    Rfrg_NPI BIGINT,
    HCPCS_CD CHAR(10),
    Zipcode BIGINT,
    Suplr_Rentl_Ind CHAR(1),
    Tot_Suplrs INT,
    Tot_Suplr_Benes FLOAT,
    Tot_Suplr_Clms INT,
    Tot_Suplr_Srvcs INT,
    Avg_Suplr_Sbmtd_Chrg FLOAT,
    Avg_Suplr_Mdcr_Alowd_Amt FLOAT,
    Avg_Suplr_Mdcr_Pymt_Amt FLOAT,
    Avg_Suplr_Mdcr_Stdzd_Amt FLOAT,
    Year INT,
    Excluded CHAR(1),
    Supplier_indicator CHAR(1),
    
    PRIMARY KEY (Rfrg_NPI, Year),
    FOREIGN KEY (Rfrg_NPI) REFERENCES Dim_NPI_Year(NPI),
    FOREIGN KEY (HCPCS_CD) REFERENCES Dim_Service(HCPCS_cd)
);

 * sqlite:////dsa/groups/casestudycf25/team02/casestudycf25t02.sqlite.db
Done.


[]

## Fact CMS Open Payment  (Ownership)

In [36]:
%%sql
CREATE TABLE Fact_CMS_Open_Payment_Ownership (
    Record_ID BIGINT,
    NPI BIGINT,
    Year INT,
    Profile_ID BIGINT,
    Change_Type VARCHAR(20),
    Applicable_Manufacturer_Payment_ID BIGINT,
    Total_Amount_Invested_USDollars INT,
    Value_of_Interest FLOAT,
    Terms_of_Interest CHAR(50),
    Payment_Publication_Date VARCHAR(50),
    PRIMARY KEY (NPI, Year),
    FOREIGN KEY (NPI) REFERENCES Dim_NPI(NPI)
);

 * sqlite:////dsa/groups/casestudycf25/team02/casestudycf25t02.sqlite.db
Done.


[]

## Fact CMS Open Payment (General)

In [38]:
%%sql
CREATE TABLE Fact_CMS_Open_Payment_General (
    Record_ID BIGINT,
    NPI BIGINT,
    Year INT,
    Profile_ID BIGINT,
    Speciality CHAR(100),
    Zip_Code BIGINT,
    Change_Type VARCHAR(20),
    Primary_Type VARCHAR(100),
    Applicable_Manufacturer_Payment_ID BIGINT,
    Total_Amount_USDollars INT,
    Type FLOAT,
    Manufacturer CHAR(100),
    Payment_Date VARCHAR(50),

    PRIMARY KEY (Record_ID, NPI, Year),
    FOREIGN KEY (NPI) REFERENCES Dim_NPI(NPI),
    FOREIGN KEY (Profile_ID) REFERENCES Dim_Provider(Profile_ID),
    FOREIGN KEY (Applicable_Manufacturer_Payment_ID) REFERENCES Fact_CMS_Open_Payment_Ownership(Applicable_Manufacturer_Payment_ID)
);

 * sqlite:////dsa/groups/casestudycf25/team02/casestudycf25t02.sqlite.db
Done.


[]

## Dim CMS Open Payments Manufacturer

In [39]:
%%sql
CREATE TABLE Dim_CMS_Open_Payments_Manufacturer (
    Manufacturing_Org_ID BIGINT PRIMARY KEY,
    Submitting_Applicable_Manufacturer_or_GPO_Name FLOAT,
    Applicable_Manufacturer_or_GPO_Making_Payment_ID FLOAT,
    Applicable_Manufacturer_or_GPO_Making_Payment_Name VARCHAR(255),
    Applicable_Manufacturer_or_GPO_Making_Payment_State VARCHAR(100),
    Applicable_Manufacturer_or_GPO_Making_Payment_Country VARCHAR(100),
    Dispute_Status_for_Publication VARCHAR(100),
    Interest_Held_by_Physician_or_Immediate_Family VARCHAR(255),
    FOREIGN KEY (Manufacturing_Org_ID) REFERENCES Fact_CMS_Open_Payment_Ownership(Applicable_Manufacturer_Payment_ID)

);

 * sqlite:////dsa/groups/casestudycf25/team02/casestudycf25t02.sqlite.db
Done.


[]